In [4]:
!nvidia-smi

Sat Sep 19 07:54:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
import transformers

print("Python/PyTorch setup")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Python/PyTorch setup
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [7]:
!pip install -q -U "transformers>=4.36.0" "trl>=0.7.0" "datasets>=2.14.0" \
    "accelerate>=0.24.0" "peft>=0.7.0" trackio

In [ ]:
# ============================================================
# HUGGING FACE SMOL COURSE — UNIT 1, EXERCISE 1
# SmolLM3 Base vs Instruct + Chat Templates + Reasoning
# ============================================================

# 1. Install required packages
!pip install -q -U "transformers>=4.36.0" "trl>=0.7.0" \
    "datasets>=2.14.0" "accelerate>=0.24.0" \
    "peft>=0.7.0" trackio

# 2. Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 3. Load SmolLM3 Base
# ============================================================

print("\n" + "=" * 60)
print("LOADING SmolLM3-3B-Base")
print("=" * 60)

base_id = "HuggingFaceTB/SmolLM3-3B-Base"

base_tokenizer = AutoTokenizer.from_pretrained(base_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded.")

# ============================================================
# 4. Load SmolLM3 Instruct
# ============================================================

print("\n" + "=" * 60)
print("LOADING SmolLM3-3B-INSTRUCT")
print("=" * 60)

instruct_id = "HuggingFaceTB/SmolLM3-3B"

instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_id)

instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Instruct model loaded.")

# ============================================================
# 5. Function for generation
# ============================================================

def generate(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(generated, skip_special_tokens=True)


# ============================================================
# 6. Compare simple prompt
# ============================================================

prompt = "Explain what machine learning is in simple words."

print("\n" + "=" * 60)
print("BASE MODEL RESPONSE")
print("=" * 60)

print(generate(
    base_model,
    base_tokenizer,
    prompt
))

print("\n" + "=" * 60)
print("INSTRUCT MODEL RESPONSE")
print("=" * 60)

print(generate(
    instruct_model,
    instruct_tokenizer,
    prompt
))

# ============================================================
# 7. Test chat template
# ============================================================

messages = [
    {
        "role": "user",
        "content": "What is artificial intelligence? Explain it simply."
    }
]

print("\n" + "=" * 60)
print("INSTRUCT CHAT TEMPLATE")
print("=" * 60)

chat_prompt = instruct_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(chat_prompt)

print("\n" + "=" * 60)
print("INSTRUCT CHAT RESPONSE")
print("=" * 60)

print(generate(
    instruct_model,
    instruct_tokenizer,
    chat_prompt
))

# ============================================================
# 8. Reasoning test
# ============================================================

reasoning_messages = [
    {
        "role": "user",
        "content": "If I have 5 shirts and buy 3 more, then give 2 away, how many shirts do I have?"
    }
]

reasoning_prompt = instruct_tokenizer.apply_chat_template(
    reasoning_messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n" + "=" * 60)
print("REASONING TEST")
print("=" * 60)

print(generate(
    instruct_model,
    instruct_tokenizer,
    reasoning_prompt,
    max_new_tokens=300
))

# ============================================================
# 9. Final status
# ============================================================

print("\n" + "=" * 60)
print("EXERCISE 1 CHECK")
print("=" * 60)

print("✓ GPU detected")
print("✓ SmolLM3-3B-Base loaded")
print("✓ SmolLM3-3B-Instruct loaded")
print("✓ Base vs Instruct comparison completed")
print("✓ Chat template tested")
print("✓ Reasoning test completed")
print("\nUNIT 1 EXERCISE 1 COMPLETE")

ENVIRONMENT
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB

LOADING SmolLM3-3B-Base


config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]